In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/fish.csv")
df.head()
print(df.columns.tolist())  

['Rank', 'Peak', 'All Time Peak', 'Actual\xa0gross', 'Adjusted\xa0gross (in 2022 dollars)', 'Artist', 'Tour title', 'Year(s)', 'Shows', 'Average gross', 'Ref.']


In [3]:
df.columns = df.columns.str.replace("\xa0", " ", regex=False)
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.replace("\xa0", " ", regex=False)

df = df.drop(columns=["Ref."])


for col in ["Peak", "All Time Peak", "Actual gross", "Tour title"]:
    df[col] = (df[col].astype(str)
                   .str.replace(r"\[.*?\]", "", regex=True)
                   .str.replace(r"[†‡*]", "",regex=True)
                   .str.strip())

for col in ["Actual gross", "Adjusted gross (in 2022 dollars)", "Average gross"]:
    df[col] = (df[col].str.replace(r"[$,]", "", regex=True).astype(float))

for col in ["Peak", "All Time Peak"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

years = df["Year(s)"].str.split(r"[\u2013-]", expand=True)
df["year_start"] = years[0].astype(int)
df["year_end"] = years[1].fillna(years[0]).astype(int)
df = df.drop(columns=["Year(s)"])

df.columns = (df.columns.str.lower()
                  .str.replace(r"[^a-z0-9]+", "_", regex=True)
                  .str.strip("_"))

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   rank                            20 non-null     int64  
 1   peak                            9 non-null      float64
 2   all_time_peak                   6 non-null      float64
 3   actual_gross                    20 non-null     float64
 4   adjusted_gross_in_2022_dollars  20 non-null     float64
 5   artist                          20 non-null     object 
 6   tour_title                      20 non-null     object 
 7   shows                           20 non-null     int64  
 8   average_gross                   20 non-null     float64
 9   year_start                      20 non-null     int64  
 10  year_end                        20 non-null     int64  
dtypes: float64(5), int64(4), object(2)
memory usage: 1.8+ KB
